In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, accuracy_score,
                             precision_score, recall_score, f1_score)
from imblearn.over_sampling import SMOTE

ROOT           = Path.cwd().parent
DATA_PROCESSED = ROOT / 'data' / 'processed'
MODELS         = ROOT / 'models'
MODELS.mkdir(exist_ok=True)


df = pd.read_csv(DATA_PROCESSED / 'bank_clean.csv')
df = pd.get_dummies(df, columns=['Geography'], drop_first=True)
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])
X = df.drop(columns=['Exited'])
y = df['Exited']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# SMOTE
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

# Scale
scaler          = StandardScaler()
X_train_scaled  = scaler.fit_transform(X_train_balanced)
X_test_scaled   = scaler.transform(X_test)

joblib.dump(scaler, MODELS / 'scaler.pkl')

print(f"Train: {X_train_scaled.shape} | Test: {X_test_scaled.shape}")
print(f"Balanced class distribution: {pd.Series(y_train_balanced).value_counts().to_dict()}")
print("All variables ready — run model cells below")

Train: (12740, 11) | Test: (2000, 11)
Balanced class distribution: {1: 6370, 0: 6370}
All variables ready — run model cells below


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib

# Train logistic regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train_balanced)

# Predict on test set
y_pred_lr    = lr.predict(X_test_scaled)
y_prob_lr    = lr.predict_proba(X_test_scaled)[:, 1]

print('=== Logistic Regression Results ===')
print(classification_report(y_test, y_pred_lr,
    target_names=['Stayed','Churned']))
print(f'ROC-AUC Score: {roc_auc_score(y_test, y_prob_lr):.4f}')

# Save model
joblib.dump(lr, '../models/logistic_regression.pkl')
print('Model saved')


=== Logistic Regression Results ===
              precision    recall  f1-score   support

      Stayed       0.87      0.79      0.83      1593
     Churned       0.41      0.55      0.47       407

    accuracy                           0.74      2000
   macro avg       0.64      0.67      0.65      2000
weighted avg       0.78      0.74      0.76      2000

ROC-AUC Score: 0.7463
Model saved


In [7]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1         
)
rf.fit(X_train_scaled, y_train_balanced)

y_pred_rf = rf.predict(X_test_scaled)
y_prob_rf = rf.predict_proba(X_test_scaled)[:, 1]

print('=== Random Forest Results ===')
print(classification_report(y_test, y_pred_rf,
    target_names=['Stayed','Churned']))
print(f'ROC-AUC Score: {roc_auc_score(y_test, y_prob_rf):.4f}')

joblib.dump(rf, '../models/random_forest.pkl')
print('Random Forest model saved')


=== Random Forest Results ===
              precision    recall  f1-score   support

      Stayed       0.91      0.86      0.88      1593
     Churned       0.54      0.67      0.60       407

    accuracy                           0.82      2000
   macro avg       0.73      0.76      0.74      2000
weighted avg       0.83      0.82      0.82      2000

ROC-AUC Score: 0.8522
Random Forest model saved


In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(name, y_true, y_pred, y_prob):
    return {
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_true, y_pred) * 100, 2),
        'Precision': round(precision_score(y_true, y_pred) * 100, 2),
        'Recall':    round(recall_score(y_true, y_pred) * 100, 2),
        'F1 Score':  round(f1_score(y_true, y_pred) * 100, 2),
        'ROC-AUC':   round(roc_auc_score(y_true, y_prob), 4),
    }

results = pd.DataFrame([
    evaluate_model('Logistic Regression', y_test, y_pred_lr, y_prob_lr),
    evaluate_model('Random Forest',       y_test, y_pred_rf, y_prob_rf),
])

print(results.to_string(index=False))
results.to_csv('../data/processed/model_comparison.csv', index=False)


              Model  Accuracy  Precision  Recall  F1 Score  ROC-AUC
Logistic Regression     74.50      40.65   55.04     46.76   0.7463
      Random Forest     81.75      54.18   66.83     59.85   0.8522
